# H6a: LR confound audit of the D-TEST depth-scaling claim

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H6a_LR_CONFOUND_AUDIT/run_experiment.py`

## Scope of this notebook
This notebook is the analysis front-end for the same deep-linear audit computed by the counterpart script. It does **not** implement a second copy of the experiment loop. Instead, it imports the script, runs the shared `run_experiment()` function, and then presents the returned results as tables and figures.

## What is measured here
- discrete LR sweeps for SGD and Muon at depths `L ∈ {2, 4, 8, 16}`
- final-loss summaries across 3 seeds
- convergence counts and fractions
- convergence-aware best-LR selection under the discrete sweeps
- a D-TEST-style replica with formula SGD LR and fixed Muon LR
- linear fits of `log(advantage)` versus depth
- temporal advantage at selected training steps

## What is *not* established here
This notebook does **not** directly measure asymptotic complexity classes, and it does **not** directly test RG / gauge-fixing observables. The present result should be read as a **deep-linear LR-confound audit under a finite LR grid**.

## Methodological correction in this first completion pass
The legacy pair silently selected “best LR” by the median over finite losses only, which could reward unstable LRs with poor convergence coverage. The shared script now uses the smallest serious fix suggested by the audit:

1. prefer the LR with the **highest convergence count** across seeds,
2. break ties by **lower median finite final loss**,
3. break remaining ties by **lower LR**.

The notebook explicitly reports where this differs from the legacy median-only selector.


In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    def Markdown(text):
        return text

    def display(*objects):
        for obj in objects:
            print(obj)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        target = candidate / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'H6a_LR_CONFOUND_AUDIT' / 'run_experiment.py'
        if target.exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing H6a run_experiment.py')


REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'H6a_LR_CONFOUND_AUDIT'
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.spec_from_file_location('h6a_lr_confound_audit', SCRIPT_PATH)
h6a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h6a)

print('Notebook-safe import complete.')
print(f'Repository root: {REPO_ROOT}')
print(f'Experiment dir:  {EXPERIMENT_DIR}')
print(f'Script path:     {SCRIPT_PATH}')
print(f'Python:          {platform.python_version()}')
print(f'NumPy:           {np.__version__}')
print(f'Pandas:          {pd.__version__}')
print(f'Working dir:     {Path.cwd().resolve()}')


## Reproducibility, runtime, and counterpart identity
The cell below records the shared configuration exposed by the script and the expected amount of work performed by one full run. This is the correct place to check whether the notebook is still analyzing the same experiment as the script.


In [ ]:
config = h6a.get_default_config()
seeds = h6a.make_seeds(config['num_seeds'], config['seed_base'], config['seed_stride'])
expected_calls = config['expected_training_calls']

config_rows = [
    ('counterpart_script', config['script_path']),
    ('experiment_dir', config['experiment_dir']),
    ('dim', config['dim']),
    ('depths', config['depths']),
    ('num_steps', config['num_steps']),
    ('momentum', config['momentum']),
    ('ns_iters', config['ns_iters']),
    ('batch_size', config['batch_size']),
    ('num_seeds', config['num_seeds']),
    ('seeds', seeds),
    ('sgd_lrs', config['sgd_lrs']),
    ('muon_lrs', config['muon_lrs']),
    ('measurement_steps', config['measurement_steps']),
    ('dtest_muon_lr', config['dtest_muon_lr']),
    ('selection_rule', config['selection_rule']),
    ('scope_note', config['scope_note']),
    ('expected_sweep_runs', expected_calls['sweep_runs']),
    ('expected_dtest_replica_runs', expected_calls['dtest_replica_runs']),
    ('expected_temporal_runs', expected_calls['temporal_runs']),
    ('expected_total_training_calls', expected_calls['total_training_calls']),
]

config_df = pd.DataFrame(config_rows, columns=['item', 'value'])
display(config_df)

print('Runtime note: a prior script execution on this machine took roughly ~70 seconds;')
print('actual wall time depends on BLAS implementation, CPU load, and notebook overhead.')


## Execute the shared experiment
This cell runs the same computation as the counterpart script by calling `run_experiment(verbose=False)`. The returned object is then reused throughout the rest of the notebook.


In [ ]:
run_started_at = datetime.now()
t0 = time.time()
results = h6a.run_experiment(verbose=False)
run_wall_time = time.time() - t0

assert results['run_counts']['actual_training_calls'] == results['run_counts']['total_training_calls']

print(f'Run started at: {run_started_at.isoformat(timespec="seconds")}')
print(f'Observed notebook wall time: {run_wall_time:.1f} s')
print(f'Actual training calls:       {results["run_counts"]["actual_training_calls"]}')
print(f'Selection differences vs legacy: {len(results["legacy_selection_differences"])}')
print(f'Swept fit slope / R^2: {results["fits"]["swept"]["slope"]:.4f} / {results["fits"]["swept"]["r2"]:.4f}')
print(f'D-TEST replica slope / R^2: {results["fits"]["dtest_replica"]["slope"]:.4f} / {results["fits"]["dtest_replica"]["r2"]:.4f}')


## Structured summaries
The next cell converts the returned result dictionary into summary tables suitable for inspection. The key methodological checks are:

- whether the convergence-aware selector changed any chosen LRs,
- whether any selected LR sits on a sweep boundary,
- whether the long-form selected-LR diagnostics expose optimizer/depth cases that need extra caution,
- how the swept-LR audit compares with the D-TEST-style replica.


In [ ]:
sweep_rows = []
for depth, depth_data in results['sweep'].items():
    for optimizer, opt_data in depth_data.items():
        selected_lr = opt_data['best']['lr']
        legacy_lr = opt_data['legacy_best']['lr']
        for row in opt_data['lr_results']:
            sweep_rows.append({
                'depth': depth,
                'optimizer': optimizer,
                'lr': row['lr'],
                'median_finite_loss': row['median_finite_loss'],
                'mean_finite_loss': row['mean_finite_loss'],
                'sem_finite_loss': row['sem_finite_loss'],
                'converged_count': row['converged_count'],
                'convergence_fraction': row['convergence_fraction'],
                'selected_lr': selected_lr,
                'legacy_selected_lr': legacy_lr,
                'is_selected': row['lr'] == selected_lr,
                'is_legacy_selected': row['lr'] == legacy_lr,
            })
sweep_df = pd.DataFrame(sweep_rows).sort_values(['optimizer', 'depth', 'lr']).reset_index(drop=True)

best_df = pd.DataFrame(results['best_by_depth']).sort_values('depth').reset_index(drop=True)
legacy_diff_df = pd.DataFrame(results['legacy_selection_differences'])

selected_lr_rows = []
for depth, depth_data in results['sweep'].items():
    for optimizer, opt_data in depth_data.items():
        best = opt_data['best']
        legacy = opt_data['legacy_best']
        selected_lr_rows.append({
            'depth': depth,
            'optimizer': optimizer,
            'selected_lr': best['lr'],
            'converged_count': best['converged_count'],
            'convergence_fraction': best['convergence_fraction'],
            'median_finite_loss': best['median_finite_loss'],
            'boundary_hit': best['boundary_hit'],
            'grid_min_lr': best['grid_min_lr'],
            'grid_max_lr': best['grid_max_lr'],
            'legacy_selected_lr': legacy['lr'],
            'legacy_converged_count': legacy['converged_count'],
            'legacy_median_finite_loss': legacy['median_finite_loss'],
            'selection_changed_vs_legacy': best['lr'] != legacy['lr'],
        })
selected_lr_df = pd.DataFrame(selected_lr_rows)
optimizer_order = {'sgd': 0, 'muon': 1}
selected_lr_df['optimizer_order'] = selected_lr_df['optimizer'].map(optimizer_order)
selected_lr_df = selected_lr_df.sort_values(['depth', 'optimizer_order']).drop(columns=['optimizer_order']).reset_index(drop=True)
boundary_df = selected_lr_df[selected_lr_df['boundary_hit']].reset_index(drop=True)

replica_rows = []
for row in results['dtest_replica']['by_depth']:
    replica_rows.append({
        'depth': row['depth'],
        'sgd_formula_lr_median': row['sgd_formula_lr_median'],
        'sgd_formula_lr_min': row['sgd_formula_lr_min'],
        'sgd_formula_lr_max': row['sgd_formula_lr_max'],
        'sgd_converged_count': row['sgd_summary']['converged_count'],
        'sgd_median_finite_loss': row['sgd_summary']['median_finite_loss'],
        'muon_lr_fixed': row['muon_summary']['lr'],
        'muon_converged_count': row['muon_summary']['converged_count'],
        'muon_median_finite_loss': row['muon_summary']['median_finite_loss'],
        'advantage': row['advantage'],
        'log_advantage': row['log_advantage'],
    })
replica_df = pd.DataFrame(replica_rows).sort_values('depth').reset_index(drop=True)

fit_df = pd.DataFrame([
    {
        'protocol': 'swept',
        'label': results['fits']['swept']['label'],
        'slope': results['fits']['swept']['slope'],
        'intercept': results['fits']['swept']['intercept'],
        'per_layer_factor': results['fits']['swept']['per_layer_factor'],
        'r2': results['fits']['swept']['r2'],
        'n_points': results['fits']['swept']['n_points'],
    },
    {
        'protocol': 'dtest_replica',
        'label': results['fits']['dtest_replica']['label'],
        'slope': results['fits']['dtest_replica']['slope'],
        'intercept': results['fits']['dtest_replica']['intercept'],
        'per_layer_factor': results['fits']['dtest_replica']['per_layer_factor'],
        'r2': results['fits']['dtest_replica']['r2'],
        'n_points': results['fits']['dtest_replica']['n_points'],
    },
])

summary_cols = [
    'depth',
    'sgd_best_lr', 'sgd_converged_count', 'sgd_convergence_fraction', 'sgd_median_finite_loss', 'sgd_boundary_hit',
    'sgd_legacy_lr', 'sgd_legacy_converged_count', 'sgd_selection_changed_vs_legacy',
    'muon_best_lr', 'muon_converged_count', 'muon_convergence_fraction', 'muon_median_finite_loss', 'muon_boundary_hit',
    'muon_legacy_lr', 'muon_legacy_converged_count', 'muon_selection_changed_vs_legacy',
    'advantage', 'log_advantage'
]

selected_lr_cols = [
    'depth', 'optimizer', 'selected_lr', 'converged_count', 'convergence_fraction', 'median_finite_loss',
    'boundary_hit', 'grid_min_lr', 'grid_max_lr', 'legacy_selected_lr', 'legacy_converged_count',
    'selection_changed_vs_legacy'
]

boundary_cols = [
    'depth', 'optimizer', 'selected_lr', 'converged_count', 'convergence_fraction', 'grid_min_lr', 'grid_max_lr'
]

display(Markdown('### Best-LR summary by depth'))
display(best_df[summary_cols])

display(Markdown('### Selected-LR diagnostics (long-form, one optimizer/depth pair per row)'))
display(selected_lr_df[selected_lr_cols])

if not boundary_df.empty:
    display(Markdown('### Boundary-hit diagnostics for selected LRs'))
    display(boundary_df[boundary_cols])
else:
    display(Markdown('### Boundary-hit diagnostics for selected LRs'))
    print('No selected LR lies on the edge of its tested sweep grid in this run.')

display(Markdown('### D-TEST-style replica summary'))
display(replica_df)

display(Markdown('### Fit summary'))
display(fit_df)

if not legacy_diff_df.empty:
    display(Markdown('### Cases where convergence-aware selection differs from the legacy median-only selector'))
    display(legacy_diff_df.sort_values(['optimizer', 'depth']).reset_index(drop=True))
else:
    display(Markdown('### Legacy selection differences'))
    print('No LR choices differed from the legacy median-over-finite-only rule in this run.')


### Interpretation of the summary tables
- The **best-LR table** is now explicit about convergence coverage, rather than silently reporting medians over whatever subset happened not to diverge.
- The **selected-LR diagnostics** table makes the actual decision inputs visible in long form: chosen LR, convergence fraction, boundary status, and the legacy choice for the same optimizer/depth pair.
- A **boundary-hit** flag means the selected LR lies on the edge of the tested grid, so the discrete sweep may still be truncating the true optimum.
- The **replica table** should be read as a protocol comparison, not as a complexity-theory result.


In [ ]:
depths = config['depths']
num_seeds = config['num_seeds']

optimizer_colors = {'sgd': 'tab:blue', 'muon': 'tab:orange'}
all_finite = sweep_df[np.isfinite(sweep_df['median_finite_loss'])]
finite_max_by_opt = all_finite.groupby('optimizer')['median_finite_loss'].max().to_dict()
sentinel_by_opt = {opt: finite_max_by_opt.get(opt, 1.0) * 5.0 for opt in ['sgd', 'muon']}

fig, axes = plt.subplots(2, len(depths), figsize=(4.1 * len(depths), 7.2), constrained_layout=True)
scatter = None

for col, depth in enumerate(depths):
    for row_idx, optimizer in enumerate(['sgd', 'muon']):
        ax = axes[row_idx, col]
        sub = sweep_df[(sweep_df['depth'] == depth) & (sweep_df['optimizer'] == optimizer)].copy().sort_values('lr')
        y_plot = []
        for _, item in sub.iterrows():
            if np.isfinite(item['median_finite_loss']):
                y_plot.append(item['median_finite_loss'])
            else:
                y_plot.append(sentinel_by_opt[optimizer])
        ax.plot(sub['lr'], y_plot, color=optimizer_colors[optimizer], alpha=0.75)
        scatter = ax.scatter(
            sub['lr'],
            y_plot,
            c=sub['convergence_fraction'],
            cmap='viridis',
            vmin=0.0,
            vmax=1.0,
            s=75,
            edgecolors='black',
            linewidths=0.4,
            zorder=3,
        )
        selected = sub[sub['is_selected']].iloc[0]
        selected_y = selected['median_finite_loss'] if np.isfinite(selected['median_finite_loss']) else sentinel_by_opt[optimizer]
        ax.scatter([selected['lr']], [selected_y], s=180, facecolors='none', edgecolors='crimson', linewidths=2.0, zorder=4)

        for (_, item), y_val in zip(sub.iterrows(), y_plot):
            ax.annotate(
                f"{int(item['converged_count'])}/{num_seeds}",
                (item['lr'], y_val),
                textcoords='offset points',
                xytext=(0, 7),
                ha='center',
                fontsize=8,
            )

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(f"{optimizer.upper()} • depth {depth}")
        ax.set_xlabel('learning rate')
        if col == 0:
            ax.set_ylabel('median finite final loss')

cbar = fig.colorbar(scatter, ax=axes, shrink=0.85)
cbar.set_label('convergence fraction across seeds')
fig.suptitle('LR sweeps by depth with explicit convergence visibility\n(circle = finite median; red ring = selected LR)', y=1.03)
plt.show()


### Reading the LR-landscape figure
Each point is one tested LR. The text above the point reports **converged seeds / total seeds**. Colors encode convergence fraction. A red ring marks the convergence-aware selected LR. If a configuration has no finite runs, it is plotted at a sentinel height above the observed finite range so that its zero-convergence status remains visible on the log-scale plot.


In [ ]:
lr_scaling_df = pd.DataFrame(results['lr_scaling']['by_depth']).sort_values('depth').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7.5, 4.8), constrained_layout=True)
ax.plot(best_df['depth'], best_df['sgd_best_lr'], '-o', label='SGD swept best LR', color='tab:blue')
ax.plot(best_df['depth'], best_df['muon_best_lr'], '-o', label='Muon swept best LR', color='tab:orange')
ax.plot(replica_df['depth'], replica_df['sgd_formula_lr_median'], '--s', label='D-TEST formula SGD LR (median)', color='tab:green')
ax.fill_between(
    replica_df['depth'],
    replica_df['sgd_formula_lr_min'],
    replica_df['sgd_formula_lr_max'],
    color='tab:green',
    alpha=0.15,
    label='D-TEST formula SGD LR range across seeds',
)

for row in best_df.itertuples(index=False):
    if row.sgd_boundary_hit:
        ax.annotate('SGD edge', (row.depth, row.sgd_best_lr), textcoords='offset points', xytext=(4, 6), fontsize=8, color='tab:blue')
    if row.muon_boundary_hit:
        ax.annotate('Muon edge', (row.depth, row.muon_best_lr), textcoords='offset points', xytext=(4, -12), fontsize=8, color='tab:orange')

ax.set_yscale('log')
ax.set_xlabel('depth')
ax.set_ylabel('learning rate')
ax.set_title('Best-LR versus depth: swept selection versus D-TEST formula baseline')
ax.legend(loc='best')
plt.show()

display(lr_scaling_df)


### Interpretation of the LR-scaling comparison
This figure asks a narrow but important question: **does the D-TEST SGD formula track the LR region preferred by the discrete sweep?** If the formula systematically drifts away from the sweep-selected SGD LR as depth increases, then part of the apparent Muon-vs-SGD depth trend may reflect protocol mismatch rather than a pure optimizer effect.


In [ ]:
swept_fit = results['fits']['swept']
dtest_fit = results['fits']['dtest_replica']

fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
ax.scatter(best_df['depth'], best_df['log_advantage'], color='tab:purple', s=70, label='Swept LR audit points')
ax.plot(swept_fit['depths'], swept_fit['predicted_log_advantages'], color='tab:purple', linewidth=2, label=f"Swept fit: slope={swept_fit['slope']:.4f}, R²={swept_fit['r2']:.4f}")

ax.scatter(replica_df['depth'], replica_df['log_advantage'], color='tab:red', marker='s', s=65, label='D-TEST replica points')
ax.plot(dtest_fit['depths'], dtest_fit['predicted_log_advantages'], color='tab:red', linestyle='--', linewidth=2, label=f"Replica fit: slope={dtest_fit['slope']:.4f}, R²={dtest_fit['r2']:.4f}")

ax.axhline(0.0, color='black', linewidth=1, alpha=0.5)
ax.set_xlabel('depth')
ax.set_ylabel('log(SGD median loss / Muon median loss)')
ax.set_title('Depth trend of the final-loss advantage under two LR-selection protocols')
ax.legend(loc='best')
plt.show()


### Interpretation of the fit figure
The vertical axis is the **log final-loss ratio at a fixed training horizon**, not a direct runtime or complexity metric. The fitted lines summarize how that ratio changes across the four tested depths in this specific deep-linear setting. With only four depth points and three seeds, the fit should be treated as descriptive rather than definitive.


In [ ]:
measurement_steps = results['temporal_advantage']['measurement_steps']
fig, axes = plt.subplots(2, 2, figsize=(10.2, 7.2), sharex=True, sharey=True, constrained_layout=True)
axes = axes.flatten()

for ax, depth_row in zip(axes, results['temporal_advantage']['by_depth']):
    step_df = pd.DataFrame([
        {
            'step': item['step'],
            'advantage': item['advantage'],
            'sgd_converged_count': item['sgd_summary']['converged_count'],
            'muon_converged_count': item['muon_summary']['converged_count'],
        }
        for item in depth_row['step_summaries']
    ])
    plot_adv = step_df['advantage'].replace([np.inf, -np.inf], np.nan)
    ax.plot(step_df['step'], plot_adv, '-o', color='tab:brown')
    ax.axhline(1.0, color='black', linewidth=1, alpha=0.5)
    ax.set_yscale('log')
    ax.set_title(
        f"depth {depth_row['depth']}\nSGD lr={depth_row['sgd_lr']:.4f}, Muon lr={depth_row['muon_lr']:.4f}"
    )
    ax.set_xlabel('training step')
    ax.set_ylabel('median loss ratio (SGD / Muon)')

plt.suptitle('Temporal advantage at selected measurement steps using the convergence-aware best LRs', y=1.02)
plt.show()


### Interpretation of the temporal-advantage figure
These panels show whether the SGD/Muon loss ratio is already present early in training or emerges only later. This remains a **fixed-horizon optimization diagnostic**, not a proof of asymptotic convergence speed.


In [ ]:
boundary_rows = []
for row in best_df.itertuples(index=False):
    if row.sgd_boundary_hit:
        boundary_rows.append(f'- Depth {row.depth}: SGD selected LR {row.sgd_best_lr:.4f} lies on the sweep boundary.')
    if row.muon_boundary_hit:
        boundary_rows.append(f'- Depth {row.depth}: Muon selected LR {row.muon_best_lr:.4f} lies on the sweep boundary.')
if not boundary_rows:
    boundary_rows.append('- No selected best LR lies on a sweep boundary in this run.')

legacy_rows = []
for item in results['legacy_selection_differences']:
    legacy_rows.append(
        f"- Depth {item['depth']}, {item['optimizer'].upper()}: convergence-aware selection chose lr={item['selected_lr']:.4f} "
        f"({item['selected_converged_count']}/{config['num_seeds']} converged) instead of the legacy lr={item['legacy_lr']:.4f} "
        f"({item['legacy_converged_count']}/{config['num_seeds']} converged)."
    )
if not legacy_rows:
    legacy_rows.append('- The convergence-aware rule matched the legacy median-only selector for all depth/optimizer pairs in this run.')

conclusion_lines = [
    '## Calibrated conclusion',
    '',
    f"In this **32x32 deep-linear regression** audit with depths `{config['depths']}`, `{config['num_seeds']}` seeds per configuration, and a discrete LR sweep, the notebook and script now evaluate the **same shared computation**.",
    '',
    '### What the current audit supports',
    '- The D-TEST-style protocol and the convergence-aware swept-LR protocol can be compared directly because they are computed from the same returned result object.',
    '- The experiment now makes convergence coverage explicit instead of silently rewarding unstable LRs with attractive finite-only medians.',
    '- The swept-LR fit and the D-TEST-style replica fit can differ materially; that difference is relevant evidence about **LR confounding under this finite sweep**.',
    '',
    '### What the current audit does **not** support on its own',
    '- It does **not** prove an asymptotic complexity-class separation.',
    '- It does **not** directly validate RG / gauge-fixing explanations.',
    '- It does **not** provide uncertainty quantification beyond the raw 3-seed summaries.',
    '',
    '### Current run highlights',
    f"- Swept-LR fit: slope = `{results['fits']['swept']['slope']:.4f}`, per-layer factor = `{results['fits']['swept']['per_layer_factor']:.4f}`, R² = `{results['fits']['swept']['r2']:.4f}`.",
    f"- D-TEST replica fit: slope = `{results['fits']['dtest_replica']['slope']:.4f}`, per-layer factor = `{results['fits']['dtest_replica']['per_layer_factor']:.4f}`, R² = `{results['fits']['dtest_replica']['r2']:.4f}`.",
    f"- Total training calls executed: `{results['run_counts']['actual_training_calls']}`.",
    f"- Observed notebook wall time in this run: `{run_wall_time:.1f}` seconds.",
    '',
    '### Remaining cautions',
    *boundary_rows,
    *legacy_rows,
    '- Only four depths and three seeds are used.',
    '- The LR grids remain discrete and may still truncate optima.',
    '- A stronger next pass would add either more seeds, uncertainty estimates, or expanded LR grids where boundary hits remain.',
]

display(Markdown('\n'.join(conclusion_lines)))
